# Module 1.5 — Error Handling

### Python Mastery · Track 1: Python Fundamentals

Programs encounter conditions they cannot handle normally: missing files, invalid input, division by zero. Python signals these with **exceptions**. This module covers catching exceptions, responding to specific ones, raising your own, defining custom exception types, and grouping related errors.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Modify the inputs to trigger different errors and observe the handling.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Use `try`/`except`/`else`/`finally` to handle failures cleanly.
2. Catch specific exception types and handle several at once.
3. Raise exceptions, including chaining with `raise ... from`.
4. Define and use custom exception classes.
5. Recognise exception groups and the `except*` syntax (Python 3.11+).

**Prerequisites:** Modules 1.1 to 1.4.

---

## Part 1 · Why Exceptions, and the Basic `try`/`except`

An exception is Python's way of saying "I cannot continue normally." If nothing handles it, the program stops and prints a traceback. The alternative to exceptions would be checking for every possible problem before acting, which is verbose and easy to get wrong. Exceptions let you write the normal path plainly and deal with failure separately.

A `try` block contains code that might fail; an `except` block runs only if a matching exception occurs.

In [ ]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return "cannot divide by zero"

print(safe_divide(10, 2))    # 5.0
print(safe_divide(10, 0))    # handled gracefully

## Part 2 · Specific Exceptions, Multiple Handlers, `else`, and `finally`

Catch the **most specific** exception you can; a bare `except:` that catches everything hides bugs. You can provide several `except` blocks, or catch several types together with a tuple.

- `else` runs only if the `try` block raised nothing.
- `finally` runs no matter what, even if an exception propagates. It is the place for cleanup.

In [ ]:
def parse_number(text):
    try:
        value = int(text)
    except ValueError:
        print(f"'{text}' is not a whole number")
        return None
    else:
        print("parsed successfully")     # only runs when no exception occurred
        return value
    finally:
        print("attempt finished")         # always runs

print(parse_number("42"))
print("---")
print(parse_number("oops"))

In [ ]:
# Catching several exception types together:
def lookup(data, key):
    try:
        return data[key]
    except (KeyError, IndexError, TypeError) as error:
        # 'as error' captures the exception object so we can inspect it.
        return f"lookup failed: {type(error).__name__}"

print(lookup({"a": 1}, "a"))     # 1
print(lookup({"a": 1}, "b"))     # KeyError
print(lookup([10, 20], 5))       # IndexError

## Part 3 · Raising and Chaining Exceptions

Use `raise` to signal an error yourself, typically when an argument is invalid. Choose the most appropriate built-in type, such as `ValueError` for a bad value or `TypeError` for a wrong type, and include a clear message.

When you catch a low-level error and raise a higher-level one, link them with `raise NewError(...) from original`. This preserves the original cause in the traceback, which aids debugging.

In [ ]:
def set_age(age):
    if not isinstance(age, int):
        raise TypeError("age must be an integer")
    if age < 0:
        raise ValueError("age cannot be negative")
    return age

print(set_age(30))
try:
    set_age(-5)
except ValueError as e:
    print("caught:", e)

In [ ]:
def load_config(text):
    try:
        return int(text)
    except ValueError as original:
        # Wrap the low-level error in a more meaningful one, preserving the cause.
        raise RuntimeError("configuration value was not a valid integer") from original

try:
    load_config("abc")
except RuntimeError as e:
    print("error:", e)
    print("caused by:", repr(e.__cause__))   # the original ValueError is retained

## Part 4 · Custom Exception Classes

For domain-specific errors, define your own exception types by subclassing `Exception`. A small hierarchy lets callers catch a broad category or a specific case. This is clearer than reusing generic types and lets users handle your library's errors precisely.

In [ ]:
class BankError(Exception):
    """Base class for all banking errors."""

class InsufficientFunds(BankError):
    """Raised when a withdrawal exceeds the balance."""

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFunds(f"tried to withdraw {amount} from {balance}")
    return balance - amount

# A caller can catch the specific error...
try:
    withdraw(100, 250)
except InsufficientFunds as e:
    print("specific:", e)

# ...or the whole category via the base class.
try:
    withdraw(100, 250)
except BankError as e:
    print("category:", type(e).__name__)

## Part 5 · Exception Groups and `except*` (Python 3.11+)

Sometimes several errors occur together, for example when validating many fields at once. An `ExceptionGroup` bundles them, and the `except*` syntax handles each type within the group. This becomes especially relevant with concurrent code in Track 5.

In [ ]:
def validate(form):
    problems = []
    if not form.get("name"):
        problems.append(ValueError("name is required"))
    if form.get("age", 0) < 0:
        problems.append(ValueError("age must not be negative"))
    if "@" not in form.get("email", ""):
        problems.append(TypeError("email looks invalid"))
    if problems:
        raise ExceptionGroup("form validation failed", problems)

bad_form = {"name": "", "age": -1, "email": "no-at-symbol"}

try:
    validate(bad_form)
except* ValueError as group:
    print("value problems:", [str(e) for e in group.exceptions])
except* TypeError as group:
    print("type problems:", [str(e) for e in group.exceptions])

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Safe division (Easy)

In [ ]:
def divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None

print(divide(8, 4))    # 2.0
print(divide(8, 0))    # None
# Experiment: return a message instead of None.

### Example 2 — Converting input with a fallback (Easy)

In [ ]:
def to_int(text, default=0):
    try:
        return int(text)
    except ValueError:
        return default

print(to_int("15"))      # 15
print(to_int("oops"))    # 0
print(to_int("oops", -1))# -1
# Experiment: pass a float-looking string such as "3.5" and observe the fallback.

### Example 3 — `else` and `finally` working together (Medium)

In [ ]:
from io import StringIO   # an in-memory text stream, used here as a stand-in for a file

def read_first_line(stream):
    try:
        line = stream.readline()
    except Exception as e:
        print("read failed:", e)
        return None
    else:
        return line.strip()
    finally:
        stream.close()                 # cleanup always happens
        print("stream closed")

result = read_first_line(StringIO("hello world\nsecond line"))
print("first line:", result)

### Example 4 — Validating and raising (Medium)

In [ ]:
def square_root(x):
    if not isinstance(x, (int, float)):
        raise TypeError("x must be a number")
    if x < 0:
        raise ValueError("x must not be negative")
    return x ** 0.5

print(square_root(16))     # 4.0
for bad in [-4, "nine"]:
    try:
        square_root(bad)
    except (ValueError, TypeError) as e:
        print(f"{bad!r}: {type(e).__name__}: {e}")

### Example 5 — A custom exception hierarchy with chaining (Difficult)

In [ ]:
class OrderError(Exception):
    """Base class for order processing errors."""

class PaymentDeclined(OrderError):
    """Raised when payment cannot be completed."""

def charge_card(amount):
    # Pretend the payment system raises a low-level error.
    raise ConnectionError("payment gateway timed out")

def place_order(amount):
    try:
        charge_card(amount)
    except ConnectionError as original:
        raise PaymentDeclined(f"could not charge {amount}") from original

try:
    place_order(99)
except OrderError as e:
    print("order failed:", e)
    print("underlying cause:", repr(e.__cause__))

### Example 6 — Collecting multiple validation errors with `except*` (Difficult)

In [ ]:
def check(values):
    errors = []
    for v in values:
        if not isinstance(v, int):
            errors.append(TypeError(f"{v!r} is not an int"))
        elif v < 0:
            errors.append(ValueError(f"{v} is negative"))
    if errors:
        raise ExceptionGroup("validation issues", errors)
    return "all valid"

try:
    check([3, -2, "x", 5, -7])
except* TypeError as g:
    print("type errors:", [str(e) for e in g.exceptions])
except* ValueError as g:
    print("value errors:", [str(e) for e in g.exceptions])

---

## Exercises

**Exercise 1 (Easy).** Write `safe_int(s)` that returns the integer value of `s`, or `None` if `s` cannot be converted.

In [ ]:
# Your solution here


**Exercise 2 (Medium).** Given the list below, build a new list containing only the values that convert to integers, silently skipping the rest.

In [ ]:
items = ["1", "two", "3", "4.5", "5"]
# Your solution here


**Exercise 3 (Medium).** Write a function `set_quantity(n)` that raises a `ValueError` with a clear message if `n` is negative, and otherwise returns `n`. Show that the error is raised for a negative input.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Define a custom exception `TemperatureError` and a function `record(temp)` that raises it if `temp` is below absolute zero (-273.15). Demonstrate catching it.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Write a function that attempts to convert a string to a float and, on failure, raises a `RuntimeError` chained from the original `ValueError` using `raise ... from`. Catch it and print both the message and the underlying cause.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
def safe_int(s):
    try:
        return int(s)
    except (ValueError, TypeError):
        return None

print(safe_int("42"), safe_int("oops"), safe_int(None))

**Solution 2**

In [ ]:
items = ["1", "two", "3", "4.5", "5"]
result = []
for item in items:
    try:
        result.append(int(item))
    except ValueError:
        continue          # skip anything that is not a clean integer
print(result)             # [1, 3, 5]

**Solution 3**

In [ ]:
def set_quantity(n):
    if n < 0:
        raise ValueError("quantity cannot be negative")
    return n

try:
    set_quantity(-3)
except ValueError as e:
    print("caught:", e)

**Solution 4**

In [ ]:
class TemperatureError(Exception):
    """Raised for physically impossible temperatures."""

def record(temp):
    if temp < -273.15:
        raise TemperatureError(f"{temp} is below absolute zero")
    return temp

try:
    record(-300)
except TemperatureError as e:
    print("caught:", e)

**Solution 5**

In [ ]:
def to_float(text):
    try:
        return float(text)
    except ValueError as original:
        raise RuntimeError(f"could not parse {text!r} as a float") from original

try:
    to_float("not-a-number")
except RuntimeError as e:
    print("error:", e)
    print("cause:", repr(e.__cause__))

---

## Summary and Key Points

- Exceptions separate the normal path from failure handling. `try` holds risky code; `except` handles a matching error.
- Catch specific types; use a tuple to handle several. `else` runs when no error occurred; `finally` always runs and is the place for cleanup.
- Raise errors with `raise`, choosing an appropriate type and a clear message. Use `raise New(...) from original` to preserve the cause.
- Define custom exceptions by subclassing `Exception`; a small hierarchy lets callers catch broadly or specifically.
- `ExceptionGroup` with `except*` (Python 3.11+) handles several errors raised together, which matters for concurrent code.

### Next module: 1.6 — Modules & Imports

The next module covers organising code across files: `import` and `from ... import`, the `__name__ == "__main__"` idiom, and a tour of useful standard-library modules.